# ⛔ VOID — RETRACTED DESIGN (R3). DO NOT RUN.This is the **v2 partner-binding** design. It was retracted for a **biological**, not technical, reason: in a real genome a strong donor at a plausible exon distance from an acceptor generally **is** its partner, so clean negatives barely exist and the forced-choice task is intrinsically degenerate. Decoys were real annotated donors but **minor** ones (median 1 supporting isoform vs 18), so "pick the partner" collapsed into "pick the better donor".See `CLAIMS.md` **R3**. Superseded by the **coupling** design — `FINDINGS.md`.

# SpliceBERT v2 — Does a frozen pre-mRNA LM encode a splice site's **partner**?

**Setup:** drop `splicebert_v2_bundle.zip` into `MyDrive/splicebert_v2/` and run all.
The bundle carries the code **and** the prebuilt dataset — no hg38 download, no GENCODE, no git clone.
The only other thing needed is your existing `SpliceBERT.1024nt` (found automatically on Drive).

---
### Why v2 exists — v1 was fatally confounded
| flaw | measured on the real v1 data |
|---|---|
| **Positional label leak** | `acceptor = left_flank`, a *constant* → true acceptor at position **200 in 100.0%** of examples |
| **Geometry oracle** | both endpoints pinned to the window edges → a model reading **zero nucleotides** scores **PMR 1.000** |
| **Length channel** | `len(seq) = 400 + exon_len` exactly → length-only adversary **0.495** |
| **Site detection suffices** | decoys were *spurious* AG/GT motifs → **oracle site detector scores PMR 1.000** |
| **Readout can't bind** | `score = f(h_a) + g(h_d) + L(len)` — no joint term |

### What v2 does differently
- **Decoys are REAL annotated splice sites** — donors that exist in the same gene but that this acceptor **never** pairs with in *any* isoform (attested alternatives excluded — those are positives). Site detection is useless *by construction*.
- **Fixed 1024nt window**, real-genome filler (never PAD) → length carries **0 bits**.
- **Acceptor offset randomised**, over a range independent of exon length.
- **Candidates permuted** by a seed from the *sequence*, never the label.
- **Bilinear readout** (can bind) + **additive readout as a falsification control that must fail**.

### The identified stratum
`adversarial` = **both** the nearest-downstream rule **and** the length prior prefer a *decoy*.
There: length prior **0.000**, nearest-downstream **0.000**, oracle site detector **= chance**.
**Any model above chance there must be using partner information.**

Dataset: **38,058** examples — train 26,563 / val 8,885 (**431 adversarial**) / test 2,610 (sealed).

## 0 · Setup — unpack the bundle

In [ ]:
import os, sys, glob, io, json, time, zipfile, subprocess
from collections import Counter

DRIVE = "/content/drive/MyDrive/splicebert_v2"
BUNDLE = f"{DRIVE}/splicebert_v2_bundle.zip"

from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists(BUNDLE):
    raise FileNotFoundError(f"Drop splicebert_v2_bundle.zip into {DRIVE}/ then re-run this cell.")

os.makedirs("/content/gfm", exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall("/content/gfm")
if "/content/gfm" not in sys.path:
    sys.path.insert(0, "/content/gfm")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())

sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
sh("pip -q install transformers")

import numpy as np
import torch
import gfmcd.splicing.partner_data as _pd

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("gfmcd :", os.path.dirname(_pd.__file__))
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 1 · Load the prebuilt dataset + locate SpliceBERT

In [ ]:
from gfmcd.splicing.partner_data import PartnerExample

def load_dataset(path):
    import gzip
    out = []
    with gzip.open(path, "rt") as f:
        for line in f:
            d = json.loads(line)
            d["candidate_donor_pos"] = tuple(d["candidate_donor_pos"])
            d["motif_donor_pos"] = tuple(d["motif_donor_pos"])
            out.append(PartnerExample(**d))
    return out

X = load_dataset("/content/gfm/partner_dataset.jsonl.gz")
S = {k: [x for x in X if x.split == k] for k in ("train", "val", "test")}
VAL = S["val"]
V_ADV = [x for x in VAL if x.metadata["stratum"] == "adversarial"]
print(f"{len(X):,} examples   " + "  ".join(f"{k}={len(v):,}" for k, v in S.items()))
print(f"val adversarial: {len(V_ADV):,}    (test = chr8,chr9 SEALED - not touched here)")

# SpliceBERT already lives on Drive from the v1 run -- find it, do not re-upload.
CANDIDATES = [
    f"{DRIVE}/SpliceBERT.1024nt",
    "/content/drive/MyDrive/splicebert_run/raw_data/models/"
    "splicebert_official_archive/extracted/models/SpliceBERT.1024nt",
]
MODEL = next((c for c in CANDIDATES if os.path.isdir(c)), None)
if MODEL is None:
    hits = glob.glob("/content/drive/MyDrive/**/SpliceBERT.1024nt", recursive=True)
    MODEL = hits[0] if hits else None
if not (MODEL and os.path.isfile(f"{MODEL}/config.json")):
    raise FileNotFoundError("SpliceBERT.1024nt not found on Drive. Set MODEL by hand.")
print("model :", MODEL)

## 2 · 🚨 LEAK GUARDRAILS — hard gate
v1 shipped because nobody ran these.

In [ ]:
pos = Counter(x.acceptor_pos for x in X)
seqlens = {len(x.sequence) for x in X}
tix = Counter(x.true_index for x in X)
share = max(pos.values()) / len(X)

checks = [
    ("acceptor position is not constant", len(pos) >= 50, f"{len(pos)} distinct (v1 had 1)"),
    ("no acceptor position dominates", share < 0.10, f"max share {share:.1%} (v1: 100%)"),
    ("window length carries 0 bits", len(seqlens) == 1, f"len(seq) in {seqlens}"),
    ("true index is not always 0", tix[0] / len(X) < 0.75, f"index-0 share {tix[0]/len(X):.1%}"),
    ("every example is a forced choice", all(len(x.candidates) >= 2 for x in X), "min 2 candidates"),
]
ok = True
for name, passed, detail in checks:
    ok = ok and passed
    print(f"{name:36s} {'PASS' if passed else 'FAIL':5s} {detail}")
if not ok:
    raise AssertionError("LEAK GUARDRAIL FAILED - do not proceed")
print("\nAll leak guardrails pass.")

## 3 · 🚨 RIVAL TABLE — the pre-registered shortcut controls
**The gate:** on `adversarial`, the oracle site detector must sit *at* chance. Only then is the task identified.

In [ ]:
from gfmcd.splicing.partner_data import LengthPrior
from gfmcd.splicing.rivals import (
    PositionOnlyRival, GeometryOracleRival, LengthPriorRival, NearestDownstreamRival,
    OracleSiteDetectorRival, CandidateIndexRival, top1, chance_level,
)

prior = LengthPrior([x.true_exon_length for x in S["train"]])   # fit on TRAIN only
mu_d = float(np.mean([x.true_donor_pos for x in S["train"]]))
mu_end = float(np.mean([len(x.sequence) - x.true_donor_pos for x in S["train"]]))
RIVALS = [
    PositionOnlyRival(mu_d), GeometryOracleRival(mu_end), LengthPriorRival(prior),
    NearestDownstreamRival(), OracleSiteDetectorRival(), CandidateIndexRival(),
]

def rival_table(exs, title):
    ch = chance_level(exs)
    print(f"\n{title}  (n={len(exs):,}, chance={ch:.3f})")
    print(f"  {'rival':22s} {'shortcut':26s} {'acc':>6s}")
    for r in RIVALS:
        a = top1(r, exs)
        flag = "   <-- ABOVE CHANCE" if a > ch + 0.05 else ""
        print(f"  {r.name:22s} {r.shortcut:26s} {a:6.3f}{flag}")
    return ch

CH_ALL = rival_table(VAL, "ALL val")
CH_ADV = rival_table(V_ADV, "ADVERSARIAL val   <-- the identified stratum")

osd = top1(OracleSiteDetectorRival(), V_ADV)
if abs(osd - CH_ADV) > 1e-6:
    raise AssertionError("oracle site detector is NOT at chance - the dataset is broken")
print(f"\nGATE PASS: perfect site detection buys nothing ({osd:.3f} == chance {CH_ADV:.3f}).")
print(f"Any model above {CH_ADV:.3f} here MUST be using partner information.")

## 4 · Precompute frozen embeddings ⚡
Backbone is frozen and tiny → embed **once**, cache the site positions only. Probes then train on the cache in seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tok = AutoTokenizer.from_pretrained(MODEL)
bb = AutoModelForMaskedLM.from_pretrained(MODEL, output_hidden_states=True).to(DEV).eval()
for p in bb.parameters():
    p.requires_grad_(False)

HID = bb.config.hidden_size
MAXC = max(len(x.candidates) for x in X)
print(f"SpliceBERT: {bb.config.num_hidden_layers}L x {bb.config.num_attention_heads}H  hidden={HID}  max candidates={MAXC}")

EMB = f"{DRIVE}/embeddings.npz"

@torch.no_grad()
def embed(exs, bs=64):
    A = np.zeros((len(exs), HID), dtype=np.float16)
    D = np.zeros((len(exs), MAXC, HID), dtype=np.float16)
    M = np.zeros((len(exs), MAXC), dtype=bool)
    t0 = time.time()
    for i in range(0, len(exs), bs):
        chunk = exs[i:i + bs]
        enc = tok([" ".join(x.sequence) for x in chunk], return_tensors="pt", padding=True).to(DEV)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            h = bb(**enc).hidden_states[-1].float()
        h = h.cpu().numpy()
        for j, x in enumerate(chunk):
            A[i + j] = h[j, x.acceptor_pos + 1]           # +1 for [CLS]
            for k, d in enumerate(x.candidates):
                D[i + j, k] = h[j, d + 1]
                M[i + j, k] = True
        if (i // bs) % 50 == 0:
            el = time.time() - t0
            print(f"   {i}/{len(exs)}  {el:.0f}s", end="\r")
    return A, D, M

if os.path.exists(EMB):
    z = dict(np.load(EMB))
    print("loaded embedding cache")
else:
    t0 = time.time()
    z = {}
    for name in ["train", "val"]:
        a, d, m = embed(S[name])
        z[f"{name}_a"], z[f"{name}_d"], z[f"{name}_m"] = a, d, m
        print(f"  {name}: {a.shape}          ")
    np.savez_compressed(EMB, **z)
    print(f"embedded in {time.time()-t0:.0f}s -> {EMB}")

## 5 · Arm 1 — partner binding
**Bilinear** (can bind) vs **Additive control** (cannot — *must* be at chance) vs **AcceptorQuery** (reads `h_a` + offset, never `h_d`).

In [ ]:
from gfmcd.splicing.partner_readout import (
    CandidateBatch, BilinearPartnerReadout, AdditiveDonorReadout,
    AcceptorQueryReadout, loss_fn, top1 as t1,
)

def make_batch(exs, a, d, m):
    off = np.zeros(m.shape, dtype=np.float32)
    lab = np.zeros(len(exs), dtype=np.int64)
    for i, x in enumerate(exs):
        for k, c in enumerate(x.candidates):
            off[i, k] = c - x.acceptor_pos
        lab[i] = x.true_index
    return CandidateBatch(
        torch.tensor(a, dtype=torch.float32), torch.tensor(d, dtype=torch.float32),
        torch.tensor(off), torch.tensor(m), torch.tensor(lab),
    )

TR = make_batch(S["train"], z["train_a"], z["train_d"], z["train_m"])
VA = make_batch(VAL, z["val_a"], z["val_d"], z["val_m"])
ADV = torch.tensor([i for i, x in enumerate(VAL) if x.metadata["stratum"] == "adversarial"])

def sub(b, idx):
    return CandidateBatch(b.acceptor[idx], b.donor[idx], b.offset[idx], b.mask[idx], b.labels[idx])

def train_probe(cls, seed, epochs=40, bs=256, lr=1e-3):
    torch.manual_seed(seed)
    model = cls(HID).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    n = TR.labels.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = sub(TR, perm[i:i + bs]).to(DEV)
            opt.zero_grad()
            loss_fn(model(b), b).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        v = VA.to(DEV)
        acc_all = t1(model(v), v).item()
        av = sub(VA, ADV).to(DEV)
        acc_adv = t1(model(av), av).item()
    return model, acc_all, acc_adv

RES, MODELS = {}, {}
for label, cls in [
    ("bilinear (hypothesis)", BilinearPartnerReadout),
    ("ADDITIVE CONTROL", AdditiveDonorReadout),
    ("acceptor-query", AcceptorQueryReadout),
]:
    runs = [train_probe(cls, s) for s in range(3)]
    MODELS[label] = runs[0][0]
    a = [r[1] for r in runs]
    d = [r[2] for r in runs]
    RES[label] = (float(np.mean(a)), float(np.std(a)), float(np.mean(d)), float(np.std(d)))
    print(f"{label:22s} all={RES[label][0]:.3f}+-{RES[label][1]:.3f}   "
          f"ADVERSARIAL={RES[label][2]:.3f}+-{RES[label][3]:.3f}")

## 6 · Verdict

In [ ]:
print(f"{'model / rival':24s} {'ALL':>10s} {'ADVERSARIAL':>13s}")
print("-" * 50)
print(f"{'chance':24s} {CH_ALL:10.3f} {CH_ADV:13.3f}")
for r in RIVALS:
    print(f"{r.name:24s} {top1(r, VAL):10.3f} {top1(r, V_ADV):13.3f}")
print("-" * 50)
for k, v in RES.items():
    print(f"{k:24s} {v[0]:10.3f} {v[2]:13.3f}")

BIL = RES["bilinear (hypothesis)"][2]
ADD = RES["ADDITIVE CONTROL"][2]
SE = (CH_ADV * (1 - CH_ADV) / len(V_ADV)) ** 0.5
PASSED = (ADD <= CH_ADV + 2 * SE) and (BIL > CH_ADV + 2 * SE)

print(f"\nchance {CH_ADV:.3f}  SE {SE:.3f}  (n={len(V_ADV)});  2-sigma band = {CH_ADV + 2*SE:.3f}")
print("=" * 62)
if ADD > CH_ADV + 2 * SE:
    print("CONTROL FAILED.")
    print("The additive readout beats chance on the adversarial stratum, but it")
    print("PROVABLY cannot express partner binding (its score is invariant to the")
    print("acceptor). So a shortcut remains. DO NOT report the bilinear number.")
elif BIL > CH_ADV + 2 * SE:
    print("PARTNER BINDING DETECTED.")
    print(f"  bilinear {BIL:.3f} > chance {CH_ADV:.3f}; additive control at chance ({ADD:.3f}).")
    print("  Site detection, position, length, distance and list order are all")
    print("  neutralised by construction -> only partner information remains.")
else:
    print("NULL RESULT (publishable).")
    print(f"  bilinear {BIL:.3f} ~ chance {CH_ADV:.3f}. SpliceBERT's frozen representation")
    print("  does NOT encode partner identity beyond site-hood, length and distance.")
    print("  v1's apparent 'partner selection' was a positional leak + site detection.")
print("=" * 62)

json.dump({
    "n": {"all": len(VAL), "adversarial": len(V_ADV)},
    "chance": {"all": CH_ALL, "adversarial": CH_ADV, "se_adversarial": SE},
    "rivals": {r.name: {"all": top1(r, VAL), "adversarial": top1(r, V_ADV)} for r in RIVALS},
    "models": {k: {"all": v[0], "all_sd": v[1], "adversarial": v[2], "adversarial_sd": v[3]}
               for k, v in RES.items()},
    "gate_passed": bool(PASSED),
}, open(f"{DRIVE}/arm1_results.json", "w"), indent=2)
print(f"\nsaved -> {DRIVE}/arm1_results.json")

## 7 · Arm 2 — causal (runs only if Arm 1 passed)
Does the acceptor's representation *read* its partner? Mask the true donor → the score should collapse. Mask a real *decoy* donor → it should not.

In [ ]:
@torch.no_grad()
def masked_accuracy(model, exs, which, bs=32):
    hits = 0
    for i in range(0, len(exs), bs):
        chunk = exs[i:i + bs]
        seqs = []
        for x in chunk:
            s = list(x.sequence)
            tgt = x.true_donor_pos if which == "true" else (
                x.real_decoy_donor_pos[0] if which == "decoy" else None)
            if tgt is not None:
                for p in range(max(0, tgt - 2), min(len(s), tgt + 2)):
                    s[p] = "N"
            seqs.append(" ".join(s))
        enc = tok(seqs, return_tensors="pt", padding=True).to(DEV)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            h = bb(**enc).hidden_states[-1].float()
        C = max(len(x.candidates) for x in chunk)
        A = torch.zeros(len(chunk), HID, device=DEV)
        D = torch.zeros(len(chunk), C, HID, device=DEV)
        M = torch.zeros(len(chunk), C, dtype=torch.bool, device=DEV)
        O = torch.zeros(len(chunk), C, device=DEV)
        L = torch.zeros(len(chunk), dtype=torch.long, device=DEV)
        for j, x in enumerate(chunk):
            A[j] = h[j, x.acceptor_pos + 1]
            for k, c in enumerate(x.candidates):
                D[j, k] = h[j, c + 1]
                M[j, k] = True
                O[j, k] = c - x.acceptor_pos
            L[j] = x.true_index
        hits += int((model(CandidateBatch(A, D, O, M, L)).argmax(-1) == L).sum().item())
    return hits / len(exs)

if not PASSED:
    print("Arm 1 did not pass its gate - Arm 2 skipped by design.")
else:
    m = MODELS["bilinear (hypothesis)"]
    hard = [x for x in V_ADV if x.real_decoy_donor_pos]
    clean = masked_accuracy(m, hard, "none")
    true_masked = masked_accuracy(m, hard, "true")
    decoy_masked = masked_accuracy(m, hard, "decoy")
    print(f"adversarial stratum (n={len(hard):,}), bilinear readout:")
    print(f"  clean              : {clean:.3f}")
    print(f"  TRUE donor masked  : {true_masked:.3f}   <- should COLLAPSE toward chance ({CH_ADV:.3f})")
    print(f"  DECOY donor masked : {decoy_masked:.3f}   <- should NOT collapse (specificity)")
    json.dump({"clean": clean, "true_masked": true_masked, "decoy_masked": decoy_masked,
               "chance": CH_ADV, "n": len(hard)},
              open(f"{DRIVE}/arm2_causal.json", "w"), indent=2)
    print(f"\nsaved -> {DRIVE}/arm2_causal.json")